# problem statement
**In this project, we analyze a transactional dataset from a UK-based online retailer covering 2010-2011. Our goal is to optimize marketing strategies and boost sales through customer segmentation. Using the K-means clustering algorithm, we transform transactional data into a customer-centric dataset, enabling us to understand distinct customer profiles and preferences. We aim to develop a recommendation system suggesting top-selling products to customers within each segment who haven't purchased those items, enhancing marketing efficacy and increasing sales.**


* **Data Cleanup & Transformation:** Ensure dataset integrity by addressing missing values, duplicates, and outliers to prepare it for optimal clustering.
* **Feature Crafting:** Generate new features from transactional data, constructing a customer-centric dataset essential for effective customer segmentation.
* **Data Prep:** Normalize features and reduce dimensionality for streamlined data, optimizing the clustering process efficiency.
* **Customer Segmentation via K-Means:** Utilize the K-means algorithm to categorize customers into groups, enabling targeted marketing and personalized strategies.
* **Cluster Analysis & Validation:** Evaluate and profile each cluster to refine marketing strategies and assess the robustness of the formed clusters.
* **Recommendation System Implementation:** Deploy a system suggesting top products to customers within the same cluster who haven't made those purchases, with the goal of enhancing sales and marketing impact.

Importing all the libraries required

In [ ]:
#ignorning the warnings
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
#for creating more complex subplots instead of using basic subplot function
import matplotlib.gridspec as gridspec
# allows you to define a colormap using a list of color specifications.
from matplotlib.colors import LinearSegmentedColormap
import plotly.graph_objects as go
from matplotlib import colors as mcolors
#this only gives basic stats and linear relation between 2 variables and used when you dont need ML model
from scipy.stats import linregress
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from yellowbrick.cluster import KElbowVisualizer, SilhouetteVisualizer
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.cluster import KMeans
# used to create formatted tables from various data structures
from tabulate import tabulate
from collections import Counter
# Matplotlib plots directly in the notebook, rather than in a separate window or file.
%matplotlib inline
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)





In [ ]:
#the csv file is not  in default ‘utf-8’
df=pd.read_csv('/kaggle/input/ecommerce-data/data.csv',encoding="ISO-8859-1")

Variable-->Description
InvoiceNo-->Code representing each unique transaction. If this code starts with letter 'c', it indicates a cancellation.
StockCode-->Code uniquely assigned to each distinct product.
Description-->Description of each product.
Quantity-->The number of units of a product in a transaction.
InvoiceDate-->The date and time of the transaction.
UnitPric-->The unit price of the product in sterling.
CustomerID-->Identifier uniquely assigned to each customer.
Country-->The country of the customer.

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
df.describe()

In [ ]:
df.describe(include='object').T

In [ ]:
df.isnull().sum()

In [ ]:
missindescrip=df['Description'].isnull().sum()/df.shape[0]*100
missincustomer=df['CustomerID'].isnull().sum()/df.shape[0]*100
print(missindescrip)
missincustomer

*****24.92% of CustomerID coloumn values are missing. 0.26% of description values are missing.Since there are ample amount of data we can just drop the rows which  has missing values*****

In [ ]:
#dropping all the rowns which have Nan values 
df.dropna(subset=['Description','CustomerID'],axis=0,how='any',inplace=True)
df

In [ ]:
df.isnull().sum()

In [ ]:
duplicate_rows=df.duplicated(keep=False)
df[duplicate_rows].sort_values(by='InvoiceNo')


In [ ]:
df.drop_duplicates(inplace=True)
df.shape[0]

In [ ]:
df['Transaction_Status'] = np.where(df['InvoiceNo'].astype(str).str.startswith('C'), 'Cancelled', 'Completed')

canceltrans = df[df['Transaction_Status'] == 'Cancelled']
canceltrans.describe().drop('CustomerID', axis=1)


In [ ]:
canceltrans.shape[0]/df.shape[0]*100


*****2.209% are the cancelled transactions*****

In [ ]:
#top 10 popular stocks
top10stock = df['StockCode'].value_counts().head(10) 
top10stock.plot(
    kind='bar',    # Change plot type to bar chart
    title='Bar Chart Example',
    xlabel='Index',
    ylabel='Values',
    color='green',  # Change bar color
    grid=True       # Display grid
)

In [ ]:
#poCategoricalst in the above graph can just be postal charges to avoide those 
unique_stock_code=df['StockCode'].unique()
unique_stock_code

In [ ]:

pd.Series(unique_stock_code).apply(lambda x:sum(c.isdigit()for c in x)).value_counts()

In [ ]:
type(unique_stock_code)

In [ ]:
anomalous_stock_code=[code for code in unique_stock_code if sum( c.isdigit() for c in code) in (0,1)]

In [ ]:
anomalous_stock_code

***now remove the anomalies***

In [ ]:
df=df[~df['StockCode'].isin(anomalous_stock_code)]

In [ ]:
#removed the anomalous stock code
df.shape


***now lets look at description column and see if thers anything wrong we can do to clean up***

In [ ]:
df['Description'].value_counts().head(50)


In [ ]:
lower_discription=df['Description'].unique()
lower_discription=[d for d in lower_discription if any(x.islower()for x in d )  ]
lower_discription

In [ ]:
#now drop unrelated descriptions
df=df[~df['Description'].isin(['Next Day Carriage','High Resolution Image'])]

In [ ]:
df.isin(['Next Day Carriage','High Resolution Image']).sum()

In [ ]:
#make all descriptions either upper or lower case.
df['Description'] = df['Description'].str.upper()
df['Description']

***now check the unitprice coloumn***

In [ ]:
df['UnitPrice'].describe()

In [ ]:
#minimum is given 0 so mean free,it might be a error or something else
sum(df['UnitPrice']==0)
#33 unitprices are given 0


In [ ]:
df=df[df['UnitPrice']>0]


In [ ]:
df['UnitPrice'].describe()

In [ ]:
#now after all this data cleaning there are many missing indexes so reset the index for a fresh index


df.reset_index( drop=True,inplace=True)
df

***Till now we have made data cleaning and intial data analysis.now it is time for "feature engineering".let us create a new customer centric data frame ad include the information related only to this helps us with better customer clustering ***

***in the customer centric Dataframe lets include RFM feature of the customer.The R is for Recency,F for frequency,M for monetary value of the customer ***

# Recency
**this helps us know about the engagement of the customer."days since purchase" lets create this feature and calculate it from latest date on a invoice for a customer**

In [ ]:
#first lets convert the invoice data to datatime variable to be able to perform operation on it
df['InvoiceDate']=pd.to_datetime(df['InvoiceDate'])
df['InvoiceDate']

In [ ]:
df['InvoiceDay'] = df['InvoiceDate'].dt.date
customer_data=df.groupby(['CustomerID'])['InvoiceDay'].max().reset_index()
#created a new dataframe for customer data and from on every cutomer related feature will nbe added
type(customer_data)




In [ ]:
customer_data['InvoiceDay']=pd.to_datetime(customer_data['InvoiceDay'])
most_recent_date=pd.to_datetime(customer_data['InvoiceDay'].max())
customer_data['day_since_last_purchase']=(most_recent_date-customer_data['InvoiceDay']).dt.days

customer_data.drop(['InvoiceDay'],axis=1,inplace=True)
customer_data


# **Frequency**
**frequency of the customers engagement.that is frequency of transactions.and no.of products  purchased ny the customer.**

In [ ]:
#with the code below we know the no of trasactions by a customer
total_trasactions=df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()

total_products=df.groupby('CustomerID')['Quantity'].sum().reset_index()

customer_data=customer_data.merge(total_trasactions,on='CustomerID').merge(total_products,on='CustomerID')
customer_data=customer_data.rename({'InvoiceNo':'Total_Transactions','Quantity':'Total_Products_Purchased'},axis=1)


In [ ]:
customer_data

# ***monetary***
***now lets calculate the amount spent by each customer and the average amount spent on each trasaction value***

In [ ]:
df['total_spend']=df['UnitPrice']*df['Quantity']

total_spend=df.groupby('CustomerID')['total_spend'].sum().reset_index()

customer_data=customer_data.merge(total_spend,on='CustomerID')
customer_data['Average_Transaction_Value']=customer_data['total_spend'] / customer_data['Total_Transactions']


In [ ]:
customer_data

# **product diversity**
***now lets take a deeper look at the customar taste in the products.this helps us know wheather the customer likes specific products or does he have a broad set of tastes.this helps us to cluster the user better by knowing the product diversity in his purchases ***

In [ ]:
unique_products_purchased=df.groupby('CustomerID')['StockCode'].nunique()
customer_data=customer_data.merge(unique_products_purchased,on='CustomerID')
customer_data=customer_data.rename(columns={'StockCode':'Unique_Products_Purchased'})
customer_data


# ***shopping patterns***
**in this step we will try to find the shopping patterns of the customers and crucial information to personalize their shopping experience.The features iam planning to intoduce "average days between shopping","favourite day of shopping","favourite hour of shopping" **

In [ ]:
df['Day_Of_Week']=df['InvoiceDate'].dt.dayofweek
df['Hour']=df['InvoiceDate'].dt.hour

In [ ]:
#average days between purchases by a customer
days_between_purchases = df.groupby('CustomerID')['InvoiceDay'].apply(lambda x: (x.diff().dropna()).apply(lambda y: y.days))
average_days_between_purchases=days_between_purchases.groupby('CustomerID').mean().reset_index()
average_days_between_purchases.rename({'InvoiceDay': 'Average_Days_Between_Purchases'},axis=1,inplace=True)

In [ ]:
#favorite shopping day of the week
favorite_shopping_day = df.groupby(['CustomerID', 'Day_Of_Week']).size().reset_index(name='Count')
favorite_shopping_day = favorite_shopping_day.loc[favorite_shopping_day.groupby('CustomerID')['Count'].idxmax()][['CustomerID', 'Day_Of_Week']]
favorite_shopping_day

In [ ]:
#favorite hour to shop
favorite_shopping_hour = df.groupby(['CustomerID', 'Hour']).size().reset_index(name='Count')
favorite_shopping_hour = favorite_shopping_hour.loc[favorite_shopping_hour.groupby('CustomerID')['Count'].idxmax()][['CustomerID', 'Hour']]
favorite_shopping_hour

In [ ]:
#merging all the data frames with main dataframe
temp=pd.merge(pd.merge(favorite_shopping_day,favorite_shopping_hour,on='CustomerID'),average_days_between_purchases, on='CustomerID')
customer_data=pd.merge(customer_data,temp,on='CustomerID')

In [ ]:
customer_data

# ***country of origin***
**now lets  take a look for the region specific shopping patterns**

In [ ]:
df['Country'].value_counts(normalize=True)

lets just make this column into a binary valued column.as in IS_UK or NOT_UK.cause majority of the data points are from the uk so its better to do it this way in order to strike that balance between accuracy and model training time

In [ ]:
# Group by CustomerID and Country to get the number of transactions per country for each customer
customer_country = df.groupby(['CustomerID', 'Country']).size().reset_index(name='Number_of_Transactions')

# Get the country with the maximum number of transactions for each customer (in case a customer has transactions from multiple countries)
customer_main_country = customer_country.sort_values('Number_of_Transactions', ascending=False).drop_duplicates('CustomerID')

# Create a binary column indicating whether the customer is from the UK or not
customer_main_country['Is_UK'] = customer_main_country['Country'].apply(lambda x: 1 if x == 'United Kingdom' else 0)

# Merge this data with our customer_data dataframe
customer_data = pd.merge(customer_data, customer_main_country[['CustomerID', 'Is_UK']], on='CustomerID', how='left')


In [ ]:
customer_data['Is_UK'].value_counts()

# **cancellation pattern**
the features iam planning to introduce are
**cancellation frequency**:we can find to tal no.of trasactions cancelled.helps in tailoring strategy.

**cancellation rate**:no.of transactions cacencelled divide by there total no.of trasactions.


by incorporating the cancellation features into the customaer data we can strive for a better prediction.

In [ ]:
total_transactions = df.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
cancelled_transactions = df[df['Transaction_Status'] == 'Cancelled']

In [ ]:
#calculate the cancelled transactions for each coustomer
cancellation_frequency = cancelled_transactions.groupby('CustomerID')['InvoiceNo'].nunique().reset_index()
cancellation_frequency.rename(columns={'InvoiceNo': 'Cancellation_Frequency'}, inplace=True)

In [ ]:
customer_data = pd.merge(customer_data, cancellation_frequency, on='CustomerID', how='left')
#for customer who hasnt had any cancelled transactions
customer_data['Cancellation_Frequency'].fillna(0, inplace=True)
customer_data['Cancellation_Rate'] = customer_data['Cancellation_Frequency'] / total_transactions['InvoiceNo']

In [ ]:
customer_data.head()

# **trends**
**In trends we take a closer look at customer spending trends in a month or accross the year as in to find out the "mean spend in a month","monthly spending standard deviation","spending trend" this tells us if the loyalty is improving with time or decreasing with time**
* Monthly_Spending_Mean
* Monthly_Spending_Std
* Spending_Trend

these are the features that are going to be introduced

In [ ]:
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month

# Calculate monthly spending for each customer
monthly_spending = df.groupby(['CustomerID', 'Year', 'Month'])['total_spend'].sum().reset_index()


In [ ]:
#calculating the mean spending in a month and stansard deviation in that spending.
seasonal_buying_patterns = monthly_spending.groupby('CustomerID')['total_spend'].agg(['mean', 'std']).reset_index()
seasonal_buying_patterns.rename(columns={'mean': 'Monthly_Spending_Mean', 'std': 'Monthly_Spending_Std'}, inplace=True)

In [ ]:
#cutomers with a single trasaction has nan std,so replace it with 0
seasonal_buying_patterns['Monthly_Spending_Std'].fillna(0, inplace=True)

In [ ]:
# Calculate Trends in Spending 
# We are using the slope of the linear trend line fitted to the customer's spending over time as an indicator of spending trends
def calculate_trend(spend_data):
    # If there are more than one data points, we calculate the trend using linear regression
    if len(spend_data) > 1:
        x = np.arange(len(spend_data))
        slope, intercept,_,_,_ = linregress(x, spend_data)
        return slope
    # If there is only one data point, no trend can be calculated, hence we return 0
    else:
        return 0

In [ ]:
# Apply the calculate_trend function to find the spending trend for each customer
spending_trends = monthly_spending.groupby('CustomerID')['total_spend'].apply(calculate_trend).reset_index()
spending_trends.rename(columns={'total_spend': 'Spending_Trend'}, inplace=True)
# Merge the new features into the customer_data dataframe
customer_data = pd.merge(customer_data, seasonal_buying_patterns, on='CustomerID')
customer_data = pd.merge(customer_data, spending_trends, on='CustomerID')


In [ ]:
customer_data.info()

In [ ]:
# Changing the data type of 'CustomerID' to string as it is a unique identifier and not used in mathematical operations
customer_data['CustomerID'] = customer_data['CustomerID'].astype(str)

# Convert data types of columns to optimal types
customer_data = customer_data.convert_dtypes()

# description of customer_data dataframe
* CustomerID>              identifier uniquely assigned to each customer, used to                                     distinguish individual customers.
* days_Since_Last_Purchase>The number of days that have passed since the customer's last purchase.
* total_Transactions>The total number of transactions made by the customer.
* total_Products_Purchased>total quantity of products purchased by the customer across all transactions.
* total_Spend>total amount of money the customer has spent across all transactions.
* Average_Transaction_Value>average value of the customer's transactions, calculated as total spend divided by the number of transactions.
* Unique_Products_Purchased>number of different products the customer has purchased.
* Average_Days_Between_Purchases>average number of days between consecutive purchases made by the customer.
* Day_Of_Week>The day of the week when the customer prefers to shop, represented numerically
* Hour>The hour of the day when the customer prefers to shop, represented in a 24-hour format
* Is_UK>A binary variable indicating whether the customer is based in the UK (1) or not (0)
* Cancellation_Frequency>The total number of transactions that the customer has cancelled.
* Cancellation_Rate>The proportion of transactions that the customer has cancelled, calculated as cancellation frequency divided by total transactions.
* Monthly_Spending_Mean>The average monthly spending of the customer
* Monthly_Spending_Std>The standard deviation of the customer's monthly spending, indicating the variability in their spending pattern.
* Spending_Trend>A numerical representation of the trend in the customer's spending over time. A positive value indicates an increasing trend, a negative value indicates a decreasing trend, and a value close to zero indicates a stable trend.

# **outlier analysis**
* we have created a dataset which focuses on customer using variety of features that gives us deeper understanding.now let us check is there are any outliers in the created data set cause outliers can effect the skewness of the data.outliers effect the model and create many more problems.
* Outliers are data points that are significantly different from the majority of other points in the dataset. These points can potentially skew the results of our analysis, especially in k-means clustering where they can significantly influence the position of the cluster centroids. Therefore, it is essential to identify and treat these outliers appropriately
* Given the multi-dimensional nature of the data,I am going to use the Isolation Forest algorithm for this task. This algorithm works well for multi-dimensional data and is computationally efficient. It isolates observations by randomly selecting a feature and then randomly selecting a split value between the maximum and minimum values of the selected feature.
**Isolation Forest>>the intution behind isolation forest is a regular datapoint is much harder to isolate then an outlier**

In [ ]:
# Initializing the IsolationForest model with a contamination parameter of 0.05
model = IsolationForest(contamination=0.05, random_state=0)

# Fitting the model on our dataset (converting DataFrame to NumPy to avoid warning)
customer_data['Outlier_Scores'] = model.fit_predict(customer_data.iloc[:, 1:].to_numpy())

# Creating a new column to identify outliers (1 for inliers and -1 for outliers)
customer_data['Is_Outlier'] = [1 if x == -1 else 0 for x in customer_data['Outlier_Scores']]
#now lets visualize the inliners and ouliners percentage
outlier_percentage = customer_data['Is_Outlier'].value_counts(normalize=True) * 100
# Plotting the percentage of inliers and outliers
plt.figure(figsize=(12, 4))
outlier_percentage.plot(kind='barh')
plt.xticks(ticks=np.arange(0, 115, 5))
plt.xlabel('Percentage (%)')
plt.ylabel('Is Outlier')

# **Inference:** 
From the above plot, we can observe that about 5% of the customers have been identified as outliers in our dataset. This percentage seems to be a reasonable proportion, not too high to lose a significant amount of data, and not too low to retain potentially noisy data points. It suggests that our isolation forest algorithm has worked well in identifying a moderate percentage of outliers.

# *Strategy:*
Considering the nature of the project (customer segmentation using clustering), it is crucial to handle these outliers to prevent them from affecting the clusters' quality significantly. Therefore, I will separate these outliers for further analysis and remove them from our main dataset to prepare it for the clustering analysis.



In [ ]:
# Separate the outliers for analysis
outliers_data = customer_data[customer_data['Is_Outlier'] == 1]

# Remove the outliers from the main dataset
customer_data_cleaned = customer_data[customer_data['Is_Outlier'] == 0]

# Drop the 'Outlier_Scores' and 'Is_Outlier' columns
customer_data_cleaned = customer_data_cleaned.drop(columns=['Outlier_Scores', 'Is_Outlier'])

# Reset the index of the cleaned data
customer_data_cleaned.reset_index(drop=True, inplace=True)
customer_data_cleaned.shape

# **correlation analysis**
Before we proceed to KMeans clustering, it's essential to check the correlation between features in our dataset. The presence of multicollinearity, where features are highly correlated, can potentially affect the clustering process by not allowing the model to learn the actual underlying patterns in the data, as the features do not provide unique information. This could lead to clusters that are not well-separated and meaningful.we can utilize dimensionality reduction techniques like PCA. These techniques help in neutralizing the effect of multicollinearity by transforming the correlated features into a new set of uncorrelated variables, preserving most of the original data's variance. This step not only enhances the quality of clusters formed but also makes the clustering process more computationally efficient.

In [ ]:
colors = ['#ff6200', '#ffcaa8', 'white', '#ffcaa8', '#ff6200']
# Define a custom colormap
my_cmap = LinearSegmentedColormap.from_list('custom_map', colors, N=256)
corr = customer_data_cleaned.drop(columns=['CustomerID']).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr,cmap=my_cmap, annot=True, center=0, fmt='.2f', linewidths=2)
plt.title('Correlation Matrix', fontsize=14)
plt.show()


Looking at the heatmap, we can see that there are some pairs of variables that have high correlations, for instance:

* Monthly_Spending_Mean and Average_Transaction_Value
* Total_Spend and Total_Products_Purchased
* Total_Transactions and Total_Spend
* Cancellation_Rate and Cancellation_Frequency
* Total_Transactions and Total_Products_Purchased

These high correlations indicate that these variables move closely together, implying a degree of multicollinearity.it might be beneficial to treat this multicollinearity possibly through dimensionality reduction techniques such as PCA to create a set of uncorrelated variables. This will help in achieving more stable clusters during the KMeans clustering process.

# feature scaling
Before we move forward with the clustering and dimensionality reduction, it's imperative to scale our features. This step holds significant importance, especially in the context of distance-based algorithms like K-means and dimensionality reduction methods like PCA.
1. For K-means Clustering: K-means relies heavily on the concept of 'distance' between data points to form clusters. When features are not on a similar scale, features with larger values can disproportionately influence the clustering outcome, potentially leading to incorrect groupings.
2. For PCA: PCA aims to find the directions where the data varies the most. When features are not scaled, those with larger values might dominate these components, not accurately reflecting the underlying patterns in the data. 
Therefore, to ensure a balanced influence on the model and to reveal the true patterns in the data, I am going to standardize our data, meaning transforming the features to have a mean of 0 and a standard deviation of 1. However, not all features require scaling. Here are the exceptions and the reasons why they are excluded:
* CustomerID: This feature is just an identifier for the customers and does not contain any meaningful information for clustering.
* Is_UK: This is a binary feature indicating whether the customer is from the UK or not. Since it already takes a value of 0 or 1, scaling it won't make any significant difference.
* Day_Of_Week: This feature represents the most frequent day of the week that the customer made transactions. Since it's a categorical feature represented by integers (1 to 7), scaling it would not be necessary.

In [ ]:
# Initialize the StandardScaler
scaler = StandardScaler()

# List of columns that don't need to be scaled
columns_to_exclude = ['CustomerID', 'Is_UK', 'Day_Of_Week']

# List of columns that need to be scaled
columns_to_scale = customer_data_cleaned.columns.difference(columns_to_exclude)

# Copy the cleaned dataset
customer_data_scaled = customer_data_cleaned.copy()

# Applying the scaler to the necessary columns in the dataset
customer_data_scaled[columns_to_scale] = scaler.fit_transform(customer_data_scaled[columns_to_scale])

# Display the first few rows of the scaled data
customer_data_scaled.head()

# **dimentionality reduction**
 **why we need feature scaling**
  * Multicollinearity Detected in the above correlation analysis
  * Better Clustering with K-means
  * Noise Reduction
  * Enhanced Visualization
  * Improved Computational Efficiency

* **pricipal component analysis**

PCA is an excellent starting point because it works well in capturing linear relationships in the data, which is particularly relevant given the multicollinearity we identified in our dataset. It allows us to reduce the number of features in our dataset while still retaining a significant amount of the information.After applying PCA, if we find that the first few components do not capture a significant amount of variance, indicating a loss of vital information, we might consider exploring other non-linear methods.I will apply PCA on all the available components and plot the cumulative variance explained by them. This process will allow me to visualize how much variance each additional principal component can explain


In [ ]:
# Setting CustomerID as the index column
customer_data_scaled.set_index('CustomerID', inplace=True)
pca = PCA().fit(customer_data_scaled)
explained_variance_ratio = pca.explained_variance_ratio_
# Calculate the Cumulative Sum of the Explained Variance
cumulative_explained_variance = np.cumsum(explained_variance_ratio)
optimal_k = 6
plt.figure(figsize=(20, 10))
# Bar chart for the explained variance of each component
barplot = sns.barplot(x=list(range(1, len(cumulative_explained_variance) + 1)),
                      y=explained_variance_ratio,
                      alpha=0.8)
lineplot, = plt.plot(range(0, len(cumulative_explained_variance)), cumulative_explained_variance,
                     marker='x', linestyle='-', color='#ff6200', linewidth=2)
# Plot optimal k value line
optimal_k_line = plt.axvline(optimal_k - 1, color='red', linestyle='--',  label=f'Optimal k value = {optimal_k}') 
# Set labels and title
plt.xlabel('Number of Components', fontsize=14)
plt.ylabel('Explained Variance', fontsize=14)
plt.title('Cumulative Variance vs. Number of Components', fontsize=18)
plt.xticks(range(0, len(cumulative_explained_variance)))
plt.legend(handles=[barplot.patches[0], lineplot, optimal_k_line],
           labels=['Explained Variance of Each Component', 'Cumulative Explained Variance', f'Optimal k value = {optimal_k}'],
           loc=(0.62, 0.1),
           frameon=True,
           framealpha=1.0,)  
# Display the variance values for both graphs on the plots
x_offset = -0.3
y_offset = 0.01
for i, (ev_ratio, cum_ev_ratio) in enumerate(zip(explained_variance_ratio, cumulative_explained_variance)):
    plt.text(i, ev_ratio, f"{ev_ratio:.2f}", ha="center", va="bottom", fontsize=10)
    if i > 0:
        plt.text(i + x_offset, cum_ev_ratio + y_offset, f"{cum_ev_ratio:.2f}", ha="center", va="bottom", fontsize=8)

plt.grid(axis='both')   
plt.show()


The plot and the cumulative explained variance values indicate how much of the total variance in the dataset is captured by each principal component, as well as the cumulative variance explained by the first n components.To choose the optimal number of components, we generally look for a point where adding another component doesn't significantly increase the cumulative explained variance, often referred to as the "elbow point" in the curve.we can see that the increase in cumulative variance starts to slow down after the 6th component (which captures about 81% of the total variance).Therefore, retaining the first 6 components might be a balanced choice

In [ ]:
# Creating a PCA object with 6 components
pca = PCA(n_components=6)

# Fitting and transforming the original data to the new PCA dataframe
customer_data_pca = pca.fit_transform(customer_data_scaled)

# Creating a new dataframe from the PCA dataframe, with columns labeled PC1, PC2, etc.
customer_data_pca = pd.DataFrame(customer_data_pca, columns=['PC'+str(i+1) for i in range(pca.n_components_)])

# Adding the CustomerID index back to the new PCA dataframe
customer_data_pca.index = customer_data_scaled.index
customer_data_pca.head()

In [ ]:
# Define a function to highlight the top 3 absolute values in each column of a dataframe
def highlight_top3(column):
    top3 = column.abs().nlargest(3).index
    return ['background-color:  #ffeacc' if i in top3 else '' for i in column.index]

# Create the PCA component DataFrame and apply the highlighting function
pc_df = pd.DataFrame(pca.components_.T, columns=['PC{}'.format(i+1) for i in range(pca.n_components_)],  
                     index=customer_data_scaled.columns)

pc_df.style.apply(highlight_top3, axis=0)

# **k-means**
K-Means is an unsupervised machine learning algorithm that clusters data into a specified number of groups (K) by minimizing the within-cluster sum-of-squares (WCSS).The algorithm iteratively assigns each data point to the nearest centroid, then updates the centroids by calculating the mean of all assigned points. The process repeats until convergence or a stopping criterion is reached.

To ascertain the optimal number of clusters (k) for segmenting customers, I will explore Silhouette Method:
The Silhouette Method is an approach to find the optimal number of clusters in a dataset by evaluating the consistency within clusters and their separation from other clusters. It computes the silhouette coefficient for each data point, which measures how similar a point is to its own cluster compared to other clusters.



To determine the silhouette coefficient for a given point i, follow these steps:

Calculate a(i): Compute the average distance between point i and all other points within its cluster.

Calculate b(i): Compute the average distance between point i and all points in the nearest cluster to its own.

Compute the silhouette coefficient, s(i), for point i using the following formula:

𝑠(𝑖)=𝑏(𝑖)−𝑎(𝑖)/max(𝑏(𝑖),𝑎(𝑖))
 
Note: The silhouette coefficient quantifies the similarity of a point to its own cluster (cohesion) relative to its separation from other clusters. This value ranges from -1 to 1, with higher values signifying that the point is well aligned with its cluster and has a low similarity to neighboring clusters.


The **silhouette score** is the average silhouette coefficient calculated for all data points in a dataset. It provides an overall assessment of the clustering quality, taking into account both cohesion within clusters and separation between clusters. A higher silhouette score indicates a better clustering configuration.





In [ ]:

def silhouette_analysis(df, start_k, stop_k, figsize=(15, 16)):
    """
    Perform Silhouette analysis for a range of k values and visualize the results.
    """

    # Set the size of the figure
    plt.figure(figsize=figsize)

    # Create a grid with (stop_k - start_k + 1) rows and 2 columns
    grid = gridspec.GridSpec(stop_k - start_k + 1, 2)

    # Assign the first plot to the first row and both columns
    first_plot = plt.subplot(grid[0, :])

    # First plot: Silhouette scores for different k values
    sns.set_palette(['darkorange'])

    silhouette_scores = []

    # Iterate through the range of k values
    for k in range(start_k, stop_k + 1):
        km = KMeans(n_clusters=k, init='k-means++', n_init=10, max_iter=100, random_state=0)
        km.fit(df)
        labels = km.predict(df)
        score = silhouette_score(df, labels)
        silhouette_scores.append(score)

    best_k = start_k + silhouette_scores.index(max(silhouette_scores))

    plt.plot(range(start_k, stop_k + 1), silhouette_scores, marker='o')
    plt.xticks(range(start_k, stop_k + 1))
    plt.xlabel('Number of clusters (k)')
    plt.ylabel('Silhouette score')
    plt.title('Average Silhouette Score for Different k Values', fontsize=15)

    # Add the optimal k value text to the plot
    optimal_k_text = f'The k value with the highest Silhouette score is: {best_k}'
    plt.text(10, 0.23, optimal_k_text, fontsize=12, verticalalignment='bottom', 
             horizontalalignment='left', bbox=dict(facecolor='#fcc36d', edgecolor='#ff6200', boxstyle='round, pad=0.5'))
             

    # Second plot (subplot): Silhouette plots for each k value
    colors = sns.color_palette("bright")

    for i in range(start_k, stop_k + 1):    
        km = KMeans(n_clusters=i, init='k-means++', n_init=10, max_iter=100, random_state=0)
        row_idx, col_idx = divmod(i - start_k, 2)

        # Assign the plots to the second, third, and fourth rows
        ax = plt.subplot(grid[row_idx + 1, col_idx])

        visualizer = SilhouetteVisualizer(km, colors=colors, ax=ax)
        visualizer.fit(df)

        # Add the Silhouette score text to the plot
        score = silhouette_score(df, km.labels_)
        ax.text(0.97, 0.02, f'Silhouette Score: {score:.2f}', fontsize=12, \
                ha='right', transform=ax.transAxes, color='red')

        ax.set_title(f'Silhouette Plot for {i} Clusters', fontsize=15)

    plt.tight_layout()
    plt.show()


In [ ]:
silhouette_analysis(customer_data_pca, 3, 12, figsize=(20, 50))


** detremine the optimal K **

To interpret silhouette plots and identify the optimal number of clusters (( k )), consider the following criteria:

1️⃣ Analyze the Silhouette Plots:

Silhouette Score Width:

Wide Widths (closer to +1): Indicate that the data points in the cluster are well separated from points in other clusters, suggesting well-defined clusters.
Narrow Widths (closer to -1): Show that data points in the cluster are not distinctly separated from other clusters, indicating poorly defined clusters.
Average Silhouette Score:

High Average Width: A cluster with a high average silhouette score indicates well-separated clusters.
Low Average Width: A cluster with a low average silhouette score indicates poor separation between clusters.
2️⃣ Uniformity in Cluster Size:

2.1 Cluster Thickness:

Uniform Thickness: Indicates that clusters have a roughly equal number of data points, suggesting a balanced clustering structure.
Variable Thickness: Signifies an imbalance in the data point distribution across clusters, with some clusters having many data points and others too few.


3️⃣ Peaks in Average Silhouette Score:
Clear Peaks: A clear peak in the average silhouette score plot for a specific ( k ) value indicates this ( k ) might be optimal.


4️⃣ Minimize Fluctuations in Silhouette Plot Widths:
Uniform Widths: Seek silhouette plots with similar widths across clusters, suggesting a more balanced and optimal clustering.
Variable Widths: Avoid wide fluctuations in silhouette plot widths, indicating that clusters are not well-defined and may vary in compactness.


5️⃣ Optimal Cluster Selection:
Maximize the Overall Average Silhouette Score: Choose the ( k ) value that gives the highest average silhouette score across all clusters, indicating well-defined clusters.
Avoid Below-Average Silhouette Scores: Ensure most clusters have above-average silhouette scores to prevent suboptimal clustering structures.


6️⃣ Visual Inspection of Silhouette Plots:
Consistent Cluster Formation: Visually inspect the silhouette plots for each ( k ) value to evaluate the consistency and structure of the formed clusters.
Cluster Compactness: Look for more compact clusters, with data points having silhouette scores closer to +1, indicating better clustering.

In [ ]:
# Apply KMeans clustering using the optimal k
kmeans = KMeans(n_clusters=3, init='k-means++', n_init=10, max_iter=100, random_state=0)
kmeans.fit(customer_data_pca)

# Get the frequency of each cluster
cluster_frequencies = Counter(kmeans.labels_)

# Create a mapping from old labels to new labels based on frequency
label_mapping = {label: new_label for new_label, (label, _) in 
                 enumerate(cluster_frequencies.most_common())}

In [ ]:
label_mapping = {v: k for k, v in {2: 1, 1: 0, 0: 2}.items()}
# Apply the mapping to get the new labels
new_labels = np.array([label_mapping[label] for label in kmeans.labels_])

# Append the new cluster labels back to the original dataset
customer_data_cleaned['cluster'] = new_labels

# Append the new cluster labels to the PCA version of the dataset
customer_data_pca['cluster'] = new_labels

In [ ]:
customer_data_cleaned.head()

# cluster analysis
This step is essential to validate the effectiveness of the clustering and to ensure that the clusters are coherent and well-separated. 

I plan to use 
* 3D Visualization of Top PCs
* Cluster Distribution Visualization

# visualization of pricipal components


In [ ]:
cluster_0 = customer_data_pca[customer_data_pca['cluster'] == 0]
cluster_1 = customer_data_pca[customer_data_pca['cluster'] == 1]
cluster_2 = customer_data_pca[customer_data_pca['cluster'] == 2]
fig = go.Figure()

# Add data points for each cluster separately and specify the color
fig.add_trace(go.Scatter3d(x=cluster_0['PC1'], y=cluster_0['PC2'], z=cluster_0['PC3'], 
                           mode='markers', marker=dict(size=5, opacity=0.4), name='Cluster 0'))
fig.add_trace(go.Scatter3d(x=cluster_1['PC1'], y=cluster_1['PC2'], z=cluster_1['PC3'], 
                           mode='markers', marker=dict(size=5, opacity=0.4), name='Cluster 1'))
fig.add_trace(go.Scatter3d(x=cluster_2['PC1'], y=cluster_2['PC2'], z=cluster_2['PC3'], 
                           mode='markers', marker=dict(size=5, opacity=0.4), name='Cluster 2'))

fig.update_layout(
    scene=dict(
        xaxis=dict(gridcolor='white', title='PC1'),
        yaxis=dict(gridcolor='white', title='PC2'),
        zaxis=dict(gridcolor='white', title='PC3'),
    ))
fig.show()


# cluster distribution percentage

In [ ]:
# Calculate the percentage of customers in each cluster
cluster_percentage = (customer_data_pca['cluster'].value_counts(normalize=True) * 100).reset_index()
cluster_percentage.columns = ['Cluster', 'Percentage']
cluster_percentage.sort_values(by='Cluster', inplace=True)


# Create a horizontal bar plot
plt.figure(figsize=(10, 4))
sns.barplot(x='Percentage', y='Cluster', data=cluster_percentage, orient='h',)

# Adding percentages on the bars
for index, value in enumerate(cluster_percentage['Percentage']):
    plt.text(value+0.5, index, f'{value:.2f}%')

plt.title('Distribution of Customers Across Clusters', fontsize=14)
plt.xticks(ticks=np.arange(0, 50, 5))
plt.xlabel('Percentage (%)')

# Show the plot
plt.show()

In [ ]:
colors = ['#e8000b', '#1ac938', '#023eff']

# cluster profiling
I am going to analyze the characteristics of each cluster to understand the distinct behaviors and preferences of different customer segments and also profile each cluster to identify the key traits that define the customers in each cluster

***radar chart approch***

First of all, I am going to create radar charts to visualize the centroid values of each cluster across different features. This can give a quick visual comparison of the profiles of different clusters.To construct the radar charts, it's essential to first compute the centroid for each cluster. This centroid represents the mean value for all features within a specific cluster.


In [ ]:
# Setting 'CustomerID' column as index and assigning it to a new dataframe
df_customer = customer_data_cleaned.set_index('CustomerID')

# Standardize the data (excluding the cluster column)
scaler = StandardScaler()
df_customer_standardized = scaler.fit_transform(df_customer.drop(columns=['cluster'], axis=1))

# Create a new dataframe with standardized values and add the cluster column back
df_customer_standardized = pd.DataFrame(df_customer_standardized, columns=df_customer.columns[:-1], index=df_customer.index)
df_customer_standardized['cluster'] = df_customer['cluster']

# Calculate the centroids of each cluster
cluster_centroids = df_customer_standardized.groupby('cluster').mean()

In [ ]:
# Function to create a radar chart
def create_radar_chart(ax, angles, data, color, cluster):
    # Plot the data and fill the area
    ax.fill(angles, data, color=color, alpha=0.4)
    ax.plot(angles, data, color=color, linewidth=2, linestyle='solid')
    
    # Add a title
    ax.set_title(f'Cluster {cluster}', size=20, color=color, y=1.1)

# Set data
labels=np.array(cluster_centroids.columns)
num_vars = len(labels)

# Compute angle of each axis
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()

# The plot is circular, so we need to "complete the loop" and append the start to the end
labels = np.concatenate((labels, [labels[0]]))
angles += angles[:1]

# Initialize the figure
fig, ax = plt.subplots(figsize=(20, 10), subplot_kw=dict(polar=True), nrows=1, ncols=3)

# Create radar chart for each cluster
for i, color in enumerate(colors):
    data = cluster_centroids.loc[i].tolist()
    data += data[:1]  # Complete the loop
    create_radar_chart(ax[i], angles, data, color, i)

# Add input data
ax[0].set_xticks(angles[:-1])
ax[0].set_xticklabels(labels[:-1])

ax[1].set_xticks(angles[:-1])
ax[1].set_xticklabels(labels[:-1])

ax[2].set_xticks(angles[:-1])
ax[2].set_xticklabels(labels[:-1])

# Add a grid
ax[0].grid(color='grey', linewidth=0.5)

# Display the plot
plt.tight_layout()
plt.show()

# rador chart analysis
***Cluster 0 (Red Chart):***
* Customers in this cluster tend to spend less, with a lower number of transactions and products purchased.
* They have a slight tendency to shop during the weekends, as indicated by the very high Day_of_Week value.
* Their spending trend is relatively stable but on the lower side, and they have a low monthly spending variation (low Monthly_Spending_Std).
* These customers have not engaged in many cancellations, showing a low cancellation frequency and rate.
* The average transaction value is on the lower side, indicating that when they do shop, they tend to spend less per transaction.

***Cluster 1 (Green Chart):***
* Customers in this cluster show a moderate level of spending, but their transactions are not very frequent, as indicated by the high Days_Since_Last_Purchase and Average_Days_Between_Purchases.
* They have a very high spending trend, indicating that their spending has been increasing over time.
* These customers prefer shopping late in the day, as indicated by the high Hour value, and they mainly reside in the UK.
* They have a tendency to cancel a moderate number of transactions, with a medium cancellation frequency and rate.
* Their average transaction value is relatively high, meaning that when they shop, they tend to make substantial purchases.

***Cluster 2 (Blue Chart):***
* Customers in this cluster are high spenders with a very high total spend, and they purchase a wide variety of unique products.
* They engage in frequent transactions, but also have a high cancellation frequency and rate.
* These customers have a very low average time between purchases, and they tend to shop early in the day (low Hour value).
* Their monthly spending shows high variability, indicating that their spending patterns might be less predictable compared to other clusters.
* Despite their high spending, they show a low spending trend, suggesting that their high spending levels might be decreasing over time.

# histogram chart approach
To validate the profiles identified from the radar charts, we can plot histograms for each feature segmented by the cluster labels. These histograms will allow us to visually inspect the distribution of feature values within each cluster

In [ ]:
# Plot histograms for each feature segmented by the clusters
features = customer_data_cleaned.columns[1:-1]
clusters = customer_data_cleaned['cluster'].unique()
clusters.sort()

# Setting up the subplots
n_rows = len(features)
n_cols = len(clusters)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 3*n_rows))

# Plotting histograms
for i, feature in enumerate(features):
    for j, cluster in enumerate(clusters):
        data = customer_data_cleaned[customer_data_cleaned['cluster'] == cluster][feature]
        axes[i, j].hist(data, bins=20, edgecolor='w', alpha=0.7)
        axes[i, j].set_title(f'Cluster {cluster} - {feature}', fontsize=15)
        axes[i, j].set_xlabel('')
        axes[i, j].set_ylabel('')

# Adjusting layout to prevent overlapping
plt.tight_layout()
plt.show()

# recommendation system
I am set to develop a recommendation system to enhance the online shopping experience. This system will suggest products to customers based on the purchasing patterns prevalent in their respective clusters. Leveraging this information, the system will craft personalized recommendations, suggesting the top three products popular within their cluster that they have not yet purchased. This not only facilitates targeted marketing strategies but also enriches the personal shopping experience, potentially boosting sales. For the outlier group, a basic approach could be to recommend random products, as a starting point to engage them.

In [ ]:
# Step 1: Extract the CustomerIDs of the outliers and remove their transactions from the main dataframe
outlier_customer_ids = outliers_data['CustomerID'].astype('float').unique()
df_filtered = df[~df['CustomerID'].isin(outlier_customer_ids)]

# Step 2: Ensure consistent data type for CustomerID across both dataframes before merging
customer_data_cleaned['CustomerID'] = customer_data_cleaned['CustomerID'].astype('float')

# Step 3: Merge the transaction data with the customer data to get the cluster information for each transaction
merged_data = df_filtered.merge(customer_data_cleaned[['CustomerID', 'cluster']], on='CustomerID', how='inner')

# Step 4: Identify the top 10 best-selling products in each cluster based on the total quantity sold
best_selling_products = merged_data.groupby(['cluster', 'StockCode', 'Description'])['Quantity'].sum().reset_index()
best_selling_products = best_selling_products.sort_values(by=['cluster', 'Quantity'], ascending=[True, False])
top_products_per_cluster = best_selling_products.groupby('cluster').head(10)

# Step 5: Create a record of products purchased by each customer in each cluster
customer_purchases = merged_data.groupby(['CustomerID', 'cluster', 'StockCode'])['Quantity'].sum().reset_index()

# Step 6: Generate recommendations for each customer in each cluster
recommendations = []
for cluster in top_products_per_cluster['cluster'].unique():
    top_products = top_products_per_cluster[top_products_per_cluster['cluster'] == cluster]
    customers_in_cluster = customer_data_cleaned[customer_data_cleaned['cluster'] == cluster]['CustomerID']
    
    for customer in customers_in_cluster:
        # Identify products already purchased by the customer
        customer_purchased_products = customer_purchases[(customer_purchases['CustomerID'] == customer) & 
                                                         (customer_purchases['cluster'] == cluster)]['StockCode'].tolist()
        
        # Find top 3 products in the best-selling list that the customer hasn't purchased yet
        top_products_not_purchased = top_products[~top_products['StockCode'].isin(customer_purchased_products)]
        top_3_products_not_purchased = top_products_not_purchased.head(3)
        
        # Append the recommendations to the list
        recommendations.append([customer, cluster] + top_3_products_not_purchased[['StockCode', 'Description']].values.flatten().tolist())

# Step 7: Create a dataframe from the recommendations list and merge it with the original customer data
recommendations_df = pd.DataFrame(recommendations, columns=['CustomerID', 'cluster', 'Rec1_StockCode', 'Rec1_Description', \
                                                 'Rec2_StockCode', 'Rec2_Description', 'Rec3_StockCode', 'Rec3_Description'])
customer_data_with_recommendations = customer_data_cleaned.merge(recommendations_df, on=['CustomerID', 'cluster'], how='right')

In [ ]:
# Display 10 random rows from the customer_data_with_recommendations dataframe
customer_data_with_recommendations.set_index('CustomerID').iloc[:, -6:].sample(10, random_state=0)